# Part 3 — FDE Mindset

*CinemaStream — The Forward Deployed Engineer's Handbook*

---

In [ ]:
# ── CinemaStream: one-time setup ──────────────────────────────────────────────
# Run this cell FIRST if you are on Google Colab or a fresh local environment.
# Skip it if you have already cloned the repo and installed requirements.
#
# !pip install -r requirements.txt
# !git clone https://github.com/YOUR_ORG/cinemastream.git
# import os; os.chdir("cinemastream")
# ─────────────────────────────────────────────────────────────────────────────
# Ensure the canonical dataset exists (deterministic; safe to re-run).
try:
    from cinemastream.scripts.generate_data import generate
    generate()
except ModuleNotFoundError:
    print("Run the clone/cd lines above first (Colab), then re-run this cell.")

## Chapters in this notebook

- [Chapter 42: The FDE Mindset — From Analyst to Data Engineer](#chapter_42_the_fde_mindset_from_analyst_to_data_engineer)
- [Chapter 43: Data Quality and Validation — Great Expectations](#chapter_43_data_quality_and_validation_great_expectations)
- [Chapter 44: ETL vs ELT — Two Architectures for Moving Data](#chapter_44_etl_vs_elt_two_architectures_for_moving_data)
- [Chapter 45: Batch vs Streaming — When Data Moves](#chapter_45_batch_vs_streaming_when_data_moves)

---

# Chapter 42: The FDE Mindset — From Analyst to Data Engineer

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### The Pipeline Mental Model

```
[Source] → [Extract] → [Validate] → [Transform] → [Load] → [Consumer]
```

### Idempotency — The Core Engineering Requirement

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime

conn = sqlite3.connect(":memory:")
conn.execute("""
CREATE TABLE IF NOT EXISTS processed_events (
    event_id       INTEGER PRIMARY KEY,   -- PK ensures upsert is safe
    user_id        INTEGER,
    watch_minutes  INTEGER,
    processed_at   TIMESTAMP
)
""")

def load_events_idempotent(events: list[dict]) -> int:
    """
    Load events — safe to call multiple times.
    Duplicate event_ids are updated (upsert), not re-inserted.
    Returns: number of rows affected.
    """
    affected = 0
    for event in events:
        conn.execute("""
        INSERT INTO processed_events (event_id, user_id, watch_minutes, processed_at)
        VALUES (:event_id, :user_id, :watch_minutes, :processed_at)
        ON CONFLICT(event_id) DO UPDATE SET
            watch_minutes = excluded.watch_minutes,
            processed_at  = excluded.processed_at
        """, {**event, "processed_at": datetime.utcnow().isoformat()})
        affected += 1
    conn.commit()
    return affected

# First run
batch1 = [
    {"event_id": 1, "user_id": 101, "watch_minutes": 90},
    {"event_id": 2, "user_id": 102, "watch_minutes": 45},
]
n1 = load_events_idempotent(batch1)
print(f"First run: {n1} events loaded")
print(pd.read_sql("SELECT * FROM processed_events", conn))

In [ ]:
# Second run with same data — idempotent, no duplicates
n2 = load_events_idempotent(batch1)
print(f"\nSecond run (same data): {n2} events processed")
count = pd.read_sql("SELECT COUNT(*) AS n FROM processed_events", conn)['n'][0]
print(f"Total rows in table: {count}  ← should still be 2")

### Validation as First-Class Code

In [ ]:
from typing import Any

def validate_events(events: list[dict]) -> tuple[list[dict], list[dict]]:
    """
    Separate valid events from invalid ones.
    Returns (valid_events, rejected_events).
    Rejection reasons are attached to the rejected events.
    """
    valid, rejected = [], []

    for event in events:
        reasons = []

        # Required fields
        if event.get("event_id") is None:
            reasons.append("missing event_id")
        if event.get("user_id") is None:
            reasons.append("missing user_id")

        # Business rules
        if event.get("watch_minutes") is not None:
            if not isinstance(event["watch_minutes"], int):
                reasons.append(f"watch_minutes must be int, got {type(event['watch_minutes'])}")
            elif event["watch_minutes"] < 0:
                reasons.append(f"watch_minutes={event['watch_minutes']} is negative")
            elif event["watch_minutes"] > 300:
                reasons.append(f"watch_minutes={event['watch_minutes']} exceeds 5 hours")

        if reasons:
            rejected.append({**event, "_rejection_reasons": reasons})
        else:
            valid.append(event)

    return valid, rejected

# Test with a dirty batch
raw_batch = [
    {"event_id": 10, "user_id": 1, "watch_minutes": 90},        # valid
    {"event_id": 11, "user_id": 2, "watch_minutes": -5},         # negative minutes
    {"event_id": None, "user_id": 3, "watch_minutes": 45},       # missing event_id
    {"event_id": 13, "user_id": 4, "watch_minutes": 350},        # exceeds max
    {"event_id": 14, "user_id": 5, "watch_minutes": 60},         # valid
]

valid_events, rejected_events = validate_events(raw_batch)

print(f"Valid: {len(valid_events)} | Rejected: {len(rejected_events)}")
print("\nRejected events:")
for e in rejected_events:
    print(f"  event_id={e.get('event_id')}: {e['_rejection_reasons']}")

### Observability — Structured Logging

In [ ]:
import logging
import json
from datetime import datetime

# Structured logging: every log entry is a JSON object, not a plain string
class StructuredLogger:
    def __init__(self, pipeline_name: str):
        self.pipeline_name = pipeline_name

    def _log(self, level: str, message: str, **context) -> None:
        entry = {
            "timestamp":  datetime.utcnow().isoformat(),
            "level":      level,
            "pipeline":   self.pipeline_name,
            "message":    message,
            **context,
        }
        print(json.dumps(entry))

    def info(self, message: str, **ctx):  self._log("INFO",  message, **ctx)
    def warn(self, message: str, **ctx):  self._log("WARN",  message, **ctx)
    def error(self, message: str, **ctx): self._log("ERROR", message, **ctx)


log = StructuredLogger("watch_events_pipeline")

def run_pipeline(batch: list[dict]) -> dict:
    log.info("pipeline_start", batch_size=len(batch))

    valid, rejected = validate_events(batch)
    log.info("validation_complete",
             valid_count=len(valid),
             rejected_count=len(rejected),
             rejection_rate=round(len(rejected)/len(batch)*100, 1))

    if rejected:
        log.warn("records_rejected",
                 count=len(rejected),
                 sample_reason=rejected[0]["_rejection_reasons"][0])

    # ... load valid records ...

    log.info("pipeline_complete", loaded=len(valid))
    return {"loaded": len(valid), "rejected": len(rejected)}

result = run_pipeline(raw_batch)
print(f"\nFinal result: {result}")

### The Retry Pattern

In [ ]:
import time
import random

def with_retry(func, max_attempts: int = 3, backoff_seconds: float = 1.0):
    """
    Execute func with exponential backoff retry.
    Raises the last exception if all attempts fail.
    """
    for attempt in range(1, max_attempts + 1):
        try:
            return func()
        except Exception as e:
            if attempt == max_attempts:
                raise  # re-raise on final attempt
            wait = backoff_seconds * (2 ** (attempt - 1))
            print(f"Attempt {attempt} failed: {e}. Retrying in {wait:.1f}s...")
            time.sleep(wait)

# Simulate a flaky API
call_count = 0
def flaky_api_call():
    global call_count
    call_count += 1
    if call_count < 3:
        raise ConnectionError(f"Simulated timeout (attempt {call_count})")
    return {"status": "ok", "fx_rates": {"USD": 1.35}}

call_count = 0
result = with_retry(flaky_api_call, max_attempts=3, backoff_seconds=0.1)
print(f"\nAPI call succeeded: {result}")

## 3. CinemaStream in Practice

In [ ]:
# The system design thinking process — before touching code

system_design = {
    "consumers": ["Rohan (email)", "Dharani (email)", "dashboard (future)"],
    "sla": "delivered by 09:00 Monday Singapore time",
    "failure_mode": "if pipeline fails, send alert to Priya not silence",
    "idempotency": "running twice same week should not send duplicate emails",
    "data_freshness": "watch events land by 23:59 Sunday UTC → extract at 00:30 Monday UTC",
    "validation_gates": [
        "total_users must be between 95 and 200 (sanity bound)",
        "mrr must be positive",
        "completion_rate between 0 and 100",
        "active_viewers > 0",
    ],
    "pipeline_steps": [
        "1. Extract metrics from PostgreSQL read replica",
        "2. Validate metric values against bounds",
        "3. Format email from template",
        "4. Send via SMTP / SendGrid",
        "5. Log success to metrics_pipeline_runs table",
    ],
}

for k, v in system_design.items():
    print(f"{k}: {v}")

In [ ]:
# The metrics extraction function — applying what we've built so far
import sqlite3
import pandas as pd
from typing import TypedDict

class WeeklyMetrics(TypedDict):
    active_premium: int
    active_basic:   int
    active_free:    int
    mrr_sgd:        float
    active_viewers: int
    avg_session_min: float
    completion_pct: float
    top_movie:      str
    inactive_paid_users: int

def extract_weekly_metrics(conn) -> WeeklyMetrics:
    """Extract the weekly business metrics from the database."""
    
    metrics_sql = """
    WITH subscriber_health AS (
        SELECT plan,
               COUNT(*) - SUM(churned) AS active
        FROM users GROUP BY plan
    ),
    engagement AS (
        SELECT COUNT(DISTINCT user_id)               AS viewers,
               ROUND(AVG(watch_minutes), 1)          AS avg_min,
               ROUND(100.0 * SUM(completed)/COUNT(*), 1) AS completion_pct
        FROM watch_events
    ),
    top_movie AS (
        SELECT m.title, COUNT(*) AS n
        FROM watch_events we JOIN movies m ON we.movie_id = m.movie_id
        GROUP BY m.movie_id ORDER BY n DESC LIMIT 1
    )
    SELECT
        (SELECT active FROM subscriber_health WHERE plan = 'Premium') AS active_premium,
        (SELECT active FROM subscriber_health WHERE plan = 'Basic')   AS active_basic,
        (SELECT active FROM subscriber_health WHERE plan = 'Free')    AS active_free,
        ROUND((SELECT SUM(CASE plan WHEN 'Premium' THEN 12.90 WHEN 'Basic' THEN 8.90 ELSE 0 END)
               FROM users WHERE churned=0), 2)                       AS mrr_sgd,
        (SELECT viewers FROM engagement)                              AS active_viewers,
        (SELECT avg_min FROM engagement)                              AS avg_session_min,
        (SELECT completion_pct FROM engagement)                       AS completion_pct,
        (SELECT title FROM top_movie)                                 AS top_movie,
        (SELECT COUNT(*) FROM users u LEFT JOIN watch_events we ON u.user_id=we.user_id
         WHERE u.plan IN ('Basic','Premium') AND u.churned=0 AND we.event_id IS NULL) AS inactive_paid_users
    """
    
    row = pd.read_sql(metrics_sql, conn).iloc[0]
    return WeeklyMetrics(
        active_premium=int(row['active_premium']),
        active_basic=int(row['active_basic']),
        active_free=int(row['active_free']),
        mrr_sgd=float(row['mrr_sgd']),
        active_viewers=int(row['active_viewers']),
        avg_session_min=float(row['avg_session_min']),
        completion_pct=float(row['completion_pct']),
        top_movie=str(row['top_movie']),
        inactive_paid_users=int(row['inactive_paid_users']),
    )

def validate_metrics(metrics: WeeklyMetrics) -> list[str]:
    """Return list of validation errors. Empty list = all good."""
    errors = []
    total_users = metrics['active_premium'] + metrics['active_basic'] + metrics['active_free']
    if not (50 <= total_users <= 10_000_000):
        errors.append(f"total_users={total_users} outside expected range [50, 10M]")
    if metrics['mrr_sgd'] <= 0:
        errors.append(f"mrr_sgd={metrics['mrr_sgd']} must be positive")
    if not (0 <= metrics['completion_pct'] <= 100):
        errors.append(f"completion_pct={metrics['completion_pct']} outside [0, 100]")
    if metrics['active_viewers'] == 0:
        errors.append("active_viewers=0 — suspicious, no watch events?")
    return errors

# Load data and run
conn = sqlite3.connect(":memory:")
for table_name, csv_file in [
    ("users",        "cinemastream/data/users.csv"),
    ("movies",       "cinemastream/data/movies.csv"),
    ("watch_events", "cinemastream/data/watch_events.csv"),
]:
    df = pd.read_csv(csv_file, encoding="utf-8")
    df.to_sql(table_name, conn, index=False, if_exists="replace")

metrics = extract_weekly_metrics(conn)
errors  = validate_metrics(metrics)

if errors:
    print("VALIDATION FAILED:")
    for e in errors:
        print(f"  - {e}")
else:
    print("Validation passed. Metrics ready to send.")
    print()
    for k, v in metrics.items():
        print(f"  {k}: {v}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
VALID_DEVICES  = {"Mobile", "TV", "Tablet", "Web"}
VALID_COUNTRIES = {"SG", "MY", "ID", "PH", "TH", "VN", "IN"}

def validate_watch_event(event: dict) -> list[str]:
    errors = []
    
    eid = event.get("event_id")
    if eid is None:
        errors.append("event_id is required")
    elif not isinstance(eid, int) or eid <= 0:
        errors.append(f"event_id must be a positive integer, got {eid!r}")
    
    mins = event.get("watch_minutes")
    if mins is None:
        errors.append("watch_minutes is required")
    elif not isinstance(mins, int) or not (1 <= mins <= 240):
        errors.append(f"watch_minutes must be 1–240, got {mins!r}")
    
    device = event.get("device")
    if device not in VALID_DEVICES:
        errors.append(f"device must be one of {VALID_DEVICES}, got {device!r}")
    
    country = event.get("country")
    if not isinstance(country, str) or len(country) != 2 or not country.isupper():
        errors.append(f"country must be 2-char uppercase, got {country!r}")
    
    return errors

# Tests
test_cases = [
    {"event_id": 1,    "watch_minutes": 90, "device": "TV",     "country": "SG"},  # valid
    {"event_id": -1,   "watch_minutes": 90, "device": "TV",     "country": "SG"},  # bad id
    {"event_id": 2,    "watch_minutes": 0,  "device": "TV",     "country": "SG"},  # zero minutes
    {"event_id": 3,    "watch_minutes": 90, "device": "Radio",  "country": "SG"},  # bad device
    {"event_id": 4,    "watch_minutes": 90, "device": "Mobile", "country": "sg"},  # lowercase country
]

for i, event in enumerate(test_cases):
    errs = validate_watch_event(event)
    status = "valid" if not errs else f"INVALID: {errs}"
    print(f"Test {i+1}: {status}")

In [ ]:
import json
from datetime import datetime

class StructuredLogger:
    def __init__(self, name):
        self.name = name
    def _log(self, level, msg, **ctx):
        print(json.dumps({"ts": datetime.utcnow().isoformat()[:19], "level": level,
                          "pipeline": self.name, "msg": msg, **ctx}))
    def info(self, msg, **ctx): self._log("INFO", msg, **ctx)
    def warn(self, msg, **ctx): self._log("WARN", msg, **ctx)
    def error(self, msg, **ctx): self._log("ERROR", msg, **ctx)

def run_weekly_metrics_pipeline(conn) -> WeeklyMetrics:
    log = StructuredLogger("weekly_metrics")
    log.info("pipeline_start")
    
    metrics = extract_weekly_metrics(conn)
    log.info("extraction_complete",
             active_premium=metrics['active_premium'],
             mrr_sgd=metrics['mrr_sgd'])
    
    errors = validate_metrics(metrics)
    if errors:
        log.error("validation_failed", errors=errors)
        raise ValueError(f"Metrics validation failed: {errors}")
    
    log.info("validation_passed")
    
    print("\n=== Weekly Metrics Report ===")
    print(f"  Active subscribers: Premium={metrics['active_premium']}, Basic={metrics['active_basic']}, Free={metrics['active_free']}")
    print(f"  Monthly MRR: S${metrics['mrr_sgd']:.2f}")
    print(f"  Viewers: {metrics['active_viewers']} | Avg session: {metrics['avg_session_min']}min | Completion: {metrics['completion_pct']}%")
    print(f"  Top movie: {metrics['top_movie']}")
    print(f"  Inactive paying users (risk): {metrics['inactive_paid_users']}")
    
    log.info("pipeline_complete")
    return metrics

run_weekly_metrics_pipeline(conn)

---

# Chapter 43: Data Quality and Validation — Great Expectations

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas

In [ ]:
# Install note: pip install great-expectations
# For this chapter we use pandas + a simplified expectations pattern
# that mirrors GX's API, then show the actual GX usage

import pandas as pd
import numpy as np
from pathlib import Path

### The Manual Expectations Pattern (What GX Encodes)

In [ ]:
class ExpectationResult:
    def __init__(self, name: str, passed: bool, details: dict):
        self.name    = name
        self.passed  = passed
        self.details = details

    def __repr__(self):
        status = "PASS" if self.passed else "FAIL"
        return f"[{status}] {self.name}: {self.details}"


def expect_column_no_nulls(df: pd.DataFrame, col: str) -> ExpectationResult:
    null_count = df[col].isnull().sum()
    return ExpectationResult(
        name=f"expect_column_values_to_not_be_null({col})",
        passed=(null_count == 0),
        details={"null_count": int(null_count), "total": len(df)},
    )

def expect_column_values_to_be_in_set(df: pd.DataFrame, col: str, valid_set: set) -> ExpectationResult:
    invalid = df[~df[col].isin(valid_set)][col]
    return ExpectationResult(
        name=f"expect_column_values_to_be_in_set({col})",
        passed=(len(invalid) == 0),
        details={"invalid_count": len(invalid), "invalid_values": invalid.unique().tolist()[:5]},
    )

def expect_column_values_to_be_between(df: pd.DataFrame, col: str, min_val, max_val) -> ExpectationResult:
    out_of_range = df[(df[col] < min_val) | (df[col] > max_val)]
    return ExpectationResult(
        name=f"expect_column_values_to_be_between({col}, {min_val}, {max_val})",
        passed=(len(out_of_range) == 0),
        details={"out_of_range_count": len(out_of_range)},
    )

def expect_column_to_be_unique(df: pd.DataFrame, col: str) -> ExpectationResult:
    dup_count = df[col].duplicated().sum()
    return ExpectationResult(
        name=f"expect_column_values_to_be_unique({col})",
        passed=(dup_count == 0),
        details={"duplicate_count": int(dup_count)},
    )

def expect_row_count_to_be_between(df: pd.DataFrame, min_n: int, max_n: int) -> ExpectationResult:
    n = len(df)
    return ExpectationResult(
        name=f"expect_table_row_count_to_be_between({min_n}, {max_n})",
        passed=(min_n <= n <= max_n),
        details={"row_count": n},
    )

In [ ]:
# Build and run a suite on the CinemaStream users dataset
DATA_DIR = Path("cinemastream/data")
users_df = pd.read_csv(DATA_DIR / "users.csv", encoding="utf-8")

suite = [
    expect_row_count_to_be_between(users_df, 50, 10_000_000),
    expect_column_no_nulls(users_df, "user_id"),
    expect_column_no_nulls(users_df, "email"),
    expect_column_values_to_be_in_set(users_df, "plan", {"Free", "Basic", "Premium"}),
    expect_column_values_to_be_in_set(users_df, "country", {"SG","MY","ID","PH","TH","VN","IN"}),
    expect_column_to_be_unique(users_df, "user_id"),
    expect_column_to_be_unique(users_df, "email"),
]

print("=== Users Dataset Validation Suite ===")
all_passed = True
for result in suite:
    print(result)
    if not result.passed:
        all_passed = False

print(f"\n{'✓ ALL PASSED' if all_passed else '✗ SUITE FAILED'}")

### Introducing Great Expectations (Actual GX API)

```python
# This shows the GX API pattern — requires: pip install great-expectations
# In a production project, run this as part of your pipeline setup

# --- GX API (do not run without installation) ---
#
# import great_expectations as gx
# from great_expectations.dataset import PandasDataset
#
# # Wrap your DataFrame as a GX Dataset
# gx_df = PandasDataset(users_df)
#
# # Define expectations — same concepts as our manual suite
# gx_df.expect_column_to_not_be_null("user_id")
# gx_df.expect_column_values_to_be_in_set("plan", ["Free", "Basic", "Premium"])
# gx_df.expect_column_values_to_be_between("watch_minutes", 1, 300)
# gx_df.expect_column_values_to_be_unique("event_id")
#
# # Run validation
# result = gx_df.validate()
# print(result["success"])        # True if all passed
# print(result["statistics"])     # pass %, evaluated count
#
# --- End GX API ---

print("GX API mirrors the manual expectations pattern shown above.")
print("Key GX advantages over manual validation:")
gx_advantages = [
    "Save expectations as JSON suite files (version-controllable)",
    "HTML data documentation generated automatically",
    "Data profiler auto-suggests expectations from a dataset",
    "Integration with Airflow, dbt, Spark, and cloud warehouses",
    "Checkpoint system: run the same suite on every pipeline execution",
]
for adv in gx_advantages:
    print(f"  - {adv}")
```

### Handling Validation Failures

In [ ]:
class ValidationSuiteResult:
    def __init__(self, results: list[ExpectationResult]):
        self.results = results
        self.passed  = all(r.passed for r in results)
        self.pass_count   = sum(1 for r in results if r.passed)
        self.fail_count   = sum(1 for r in results if not r.passed)
        self.pass_rate_pct = round(100 * self.pass_count / len(results), 1) if results else 0.0

    def failed(self) -> list[ExpectationResult]:
        return [r for r in self.results if not r.passed]

    def summary(self) -> str:
        status = "PASSED" if self.passed else "FAILED"
        return (f"Suite {status}: {self.pass_count}/{len(self.results)} expectations passed "
                f"({self.pass_rate_pct}%)")

def run_validation_suite(df: pd.DataFrame, expectations: list) -> ValidationSuiteResult:
    results = [exp(df) if callable(exp) else exp for exp in expectations]
    return ValidationSuiteResult(results)


# Inject a bad row to trigger failures
dirty_users = users_df.copy()
dirty_users.loc[100] = [101, "Test User", "test@example.com", "XX", "en", "Gold", "2024-01-01", 0]

suite_on_dirty = run_validation_suite(dirty_users, [
    lambda df: expect_row_count_to_be_between(df, 50, 100),
    lambda df: expect_column_values_to_be_in_set(df, "plan", {"Free","Basic","Premium"}),
    lambda df: expect_column_values_to_be_in_set(df, "country", {"SG","MY","ID","PH","TH","VN","IN"}),
    lambda df: expect_column_to_be_unique(df, "user_id"),
])

print("=== Dirty Dataset Validation ===")
print(suite_on_dirty.summary())
print("\nFailures:")
for f in suite_on_dirty.failed():
    print(f"  {f}")

### Using GX Expectations as Pipeline Gates

In [ ]:
def validated_load(df: pd.DataFrame, target_table: str, conn) -> None:
    """
    Load df into target_table only if it passes all validation expectations.
    Raises ValueError with failure details if validation fails.
    """
    expectations = [
        lambda df: expect_row_count_to_be_between(df, 1, 10_000_000),
        lambda df: expect_column_no_nulls(df, "user_id"),
        lambda df: expect_column_values_to_be_in_set(df, "plan", {"Free","Basic","Premium"}),
        lambda df: expect_column_values_to_be_between(df, "user_id", 1, 10_000_000),
    ]

    suite_result = run_validation_suite(df, expectations)

    if not suite_result.passed:
        failures = "\n".join(str(f) for f in suite_result.failed())
        raise ValueError(
            f"Data quality gate FAILED for {target_table}:\n{failures}"
        )

    # Only reaches here if all expectations passed
    df.to_sql(target_table, conn, if_exists="replace", index=False)
    print(f"Loaded {len(df)} rows to {target_table} — all {suite_result.pass_count} expectations passed")

import sqlite3
conn2 = sqlite3.connect(":memory:")

# Clean data — should load
try:
    validated_load(users_df, "users_validated", conn2)
except ValueError as e:
    print(f"BLOCKED: {e}")

# Dirty data — should be blocked
try:
    validated_load(dirty_users, "users_validated", conn2)
except ValueError as e:
    print(f"\nBLOCKED (expected): {e}")

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("cinemastream/data")

# Load all three source tables
users_df  = pd.read_csv(DATA_DIR / "users.csv",        encoding="utf-8", parse_dates=["signup_date"])
movies_df = pd.read_csv(DATA_DIR / "movies.csv",        encoding="utf-8")
events_df = pd.read_csv(DATA_DIR / "watch_events.csv",  encoding="utf-8", parse_dates=["watch_started"])

# Define expectation suites per table
USERS_SUITE = [
    lambda df: expect_row_count_to_be_between(df, 50, 10_000_000),
    lambda df: expect_column_no_nulls(df, "user_id"),
    lambda df: expect_column_no_nulls(df, "email"),
    lambda df: expect_column_to_be_unique(df, "user_id"),
    lambda df: expect_column_to_be_unique(df, "email"),
    lambda df: expect_column_values_to_be_in_set(df, "plan", {"Free","Basic","Premium"}),
    lambda df: expect_column_values_to_be_in_set(df, "country", {"SG","MY","ID","PH","TH","VN","IN"}),
    lambda df: expect_column_values_to_be_in_set(df, "churned", {0, 1, True, False}),
]

MOVIES_SUITE = [
    lambda df: expect_row_count_to_be_between(df, 5, 100_000),
    lambda df: expect_column_no_nulls(df, "movie_id"),
    lambda df: expect_column_to_be_unique(df, "movie_id"),
    lambda df: expect_column_no_nulls(df, "title"),
    lambda df: expect_column_values_to_be_between(df, "runtime_min", 1, 400),
    lambda df: expect_column_values_to_be_in_set(df, "genre",
        {"Drama","Action","Comedy","Thriller","Romance","Documentary"}),
]

EVENTS_SUITE = [
    lambda df: expect_row_count_to_be_between(df, 1, 500_000_000),
    lambda df: expect_column_no_nulls(df, "event_id"),
    lambda df: expect_column_no_nulls(df, "user_id"),
    lambda df: expect_column_no_nulls(df, "movie_id"),
    lambda df: expect_column_to_be_unique(df, "event_id"),
    lambda df: expect_column_values_to_be_between(df, "watch_minutes", 1, 300),
    lambda df: expect_column_values_to_be_in_set(df, "device", {"Mobile","TV","Tablet","Web"}),
    lambda df: expect_column_values_to_be_in_set(df, "country", {"SG","MY","ID","PH","TH","VN","IN"}),
]

# Run all suites and produce a morning report
tables = [
    ("users",        users_df,  USERS_SUITE),
    ("movies",       movies_df, MOVIES_SUITE),
    ("watch_events", events_df, EVENTS_SUITE),
]

print("=== CinemaStream Data Quality Morning Report ===")
print(f"Run time: {datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
print()

pipeline_can_proceed = True
for table_name, df, suite in tables:
    result = run_validation_suite(df, suite)
    status = "PASS" if result.passed else "FAIL"
    print(f"[{status}] {table_name}: {result.pass_count}/{len(suite)} expectations passed ({result.pass_rate_pct}%)")
    
    if not result.passed:
        pipeline_can_proceed = False
        for failure in result.failed():
            print(f"       +- {failure}")

print()
if pipeline_can_proceed:
    print("All tables passed. Pipeline can proceed.")
else:
    print("PIPELINE BLOCKED — fix data quality issues before loading to production.")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
movies_df = pd.read_csv("cinemastream/data/movies.csv", encoding="utf-8")

movies_suite = [
    lambda df: expect_column_no_nulls(df, "movie_id"),
    lambda df: expect_column_to_be_unique(df, "movie_id"),
    lambda df: expect_column_values_to_be_between(df, "runtime_min", 30, 400),
    lambda df: expect_column_values_to_be_between(df, "release_year", 1950, 2025),
    lambda df: expect_column_values_to_be_in_set(df, "genre",
        {"Drama","Action","Comedy","Thriller","Romance","Documentary"}),
]

result = run_validation_suite(movies_df, movies_suite)
print(result.summary())
for r in result.results:
    print(f"  {r}")

In [ ]:
events_df = pd.read_csv("cinemastream/data/watch_events.csv", encoding="utf-8")

# Inject the bug
dirty_events = events_df.copy()
dirty_events.loc[dirty_events.index[:5], "watch_minutes"] = 0

suite = [
    lambda df: expect_column_values_to_be_between(df, "watch_minutes", 1, 300),
]

result = run_validation_suite(dirty_events, suite)
print(result.summary())
for r in result.results:
    print(f"  {r}")

---

# Chapter 44: ETL vs ELT — Two Architectures for Moving Data

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install numpy pandas

### The ETL Pattern in Python

In [ ]:
import pandas as pd
import sqlite3
from pathlib import Path
from datetime import datetime
import json

# ETL Example: Extract from CSV, Transform in Python, Load to SQLite
# In production: Extract from Kafka/API, Transform in Spark, Load to Redshift

def extract_watch_events(csv_path: str) -> pd.DataFrame:
    """Step E: extract raw data from source."""
    df = pd.read_csv(csv_path, encoding="utf-8", parse_dates=["watch_started"])
    print(f"[Extract] {len(df)} raw rows read from {csv_path}")
    return df

def transform_watch_events(raw_df: pd.DataFrame) -> pd.DataFrame:
    """
    Step T: transform data BEFORE loading.
    - Drop rows with invalid watch_minutes
    - Add session_bucket derived column
    - Standardise device names
    - Convert timestamps to UTC
    """
    import numpy as np
    
    df = raw_df.copy()
    
    # Drop invalid
    before = len(df)
    df = df[df["watch_minutes"].between(1, 300)]
    print(f"[Transform] Dropped {before - len(df)} out-of-range rows")
    
    # Add session_bucket
    conditions = [df["watch_minutes"] < 30, df["watch_minutes"].between(30, 89)]
    buckets    = ["short", "medium"]
    df["session_bucket"] = np.select(conditions, buckets, default="long")
    
    # Standardise device names (lowercase)
    df["device_clean"] = df["device"].str.strip().str.title()
    
    print(f"[Transform] {len(df)} rows ready to load")
    return df

def load_watch_events(df: pd.DataFrame, conn) -> None:
    """Step L: load transformed data to destination."""
    df.to_sql("watch_events_clean", conn, if_exists="replace", index=False)
    count = pd.read_sql("SELECT COUNT(*) AS n FROM watch_events_clean", conn)['n'][0]
    print(f"[Load] {count} rows written to watch_events_clean")

# Run the ETL pipeline
raw_df    = extract_watch_events("cinemastream/data/watch_events.csv")
clean_df  = transform_watch_events(raw_df)
conn = sqlite3.connect(":memory:")
load_watch_events(clean_df, conn)

print("\nSample of loaded clean data:")
result = pd.read_sql(
    "SELECT event_id, watch_minutes, session_bucket, device_clean FROM watch_events_clean LIMIT 4",
    conn
)
print(result)

### The ELT Pattern

In [ ]:
# ELT: Load raw data first, then transform WITH SQL inside the warehouse

def extract_raw(csv_path: str) -> pd.DataFrame:
    """Step E: extract raw data — minimal processing."""
    df = pd.read_csv(csv_path, encoding="utf-8")
    print(f"[Extract] {len(df)} raw rows from {csv_path}")
    return df

def load_raw(df: pd.DataFrame, table_name: str, conn) -> None:
    """Step L: load RAW data as-is (no transformation yet)."""
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"[Load] Raw {table_name}: {len(df)} rows loaded to warehouse")

def transform_with_sql(conn) -> None:
    """Step T: transform using SQL INSIDE the warehouse."""
    conn.execute("""
    CREATE TABLE IF NOT EXISTS watch_events_transformed AS
    SELECT
        event_id,
        user_id,
        movie_id,
        watch_started,
        watch_minutes,
        completed,
        TRIM(device) AS device_clean,
        country,
        CASE
            WHEN watch_minutes < 30 THEN 'short'
            WHEN watch_minutes BETWEEN 30 AND 89 THEN 'medium'
            ELSE 'long'
        END AS session_bucket
    FROM raw_watch_events
    WHERE watch_minutes BETWEEN 1 AND 300
    """)
    conn.commit()
    count = pd.read_sql("SELECT COUNT(*) AS n FROM watch_events_transformed", conn)['n'][0]
    print(f"[Transform] watch_events_transformed: {count} rows created by SQL in warehouse")

# Run ELT
elt_conn = sqlite3.connect(":memory:")
raw_df = extract_raw("cinemastream/data/watch_events.csv")
load_raw(raw_df, "raw_watch_events", elt_conn)
transform_with_sql(elt_conn)

# Key advantage: raw data is always preserved
raw_count  = pd.read_sql("SELECT COUNT(*) AS n FROM raw_watch_events", elt_conn)['n'][0]
cln_count  = pd.read_sql("SELECT COUNT(*) AS n FROM watch_events_transformed", elt_conn)['n'][0]
print(f"\nRaw data preserved: {raw_count} rows | Transformed: {cln_count} rows")
print("Raw data is always available for re-processing or debugging.")

### Re-processing: Why ELT Wins for Iteration

In [ ]:
# The key advantage of ELT: you can re-run transformations without re-extracting
# Suppose business rules change: "short" is now < 20 minutes (was < 30)
def update_transform(conn) -> None:
    conn.execute("DROP TABLE IF EXISTS watch_events_v2")
    conn.execute("""
    CREATE TABLE watch_events_v2 AS
    SELECT
        *,
        CASE
            WHEN watch_minutes < 20 THEN 'short'     -- changed from 30 → 20
            WHEN watch_minutes BETWEEN 20 AND 89 THEN 'medium'
            ELSE 'long'
        END AS session_bucket_v2
    FROM raw_watch_events
    WHERE watch_minutes BETWEEN 1 AND 300
    """)
    conn.commit()
    print("[Re-transform] watch_events_v2 created with updated business rules")

update_transform(elt_conn)

comparison = pd.read_sql("""
SELECT 
    we_t.session_bucket  AS v1_bucket,
    we_v2.session_bucket_v2 AS v2_bucket,
    COUNT(*) AS session_count
FROM watch_events_transformed we_t
JOIN watch_events_v2 we_v2 ON we_t.event_id = we_v2.event_id
GROUP BY v1_bucket, v2_bucket
ORDER BY v1_bucket
""", elt_conn)
print("\nBucket comparison v1 vs v2:")
print(comparison)

## 3. CinemaStream in Practice

In [ ]:
# The CinemaStream pipeline architecture comparison

etl_architecture = {
    "name": "Current ETL (Python-based)",
    "steps": [
        "1. Kafka consumer reads watch_events in micro-batches",
        "2. Python process validates + transforms (30 rules in Python code)",
        "3. Transformed rows loaded to BigQuery",
    ],
    "pros": [
        "Transformation logic in Python (more expressive for complex rules)",
        "Only clean data enters the warehouse (lower storage cost)",
    ],
    "cons": [
        "Raw data not preserved — bugs in transformation logic cause permanent data loss",
        "Rule changes require reprocessing from the Kafka offset (may not be possible)",
        "Python code is harder to audit than SQL for business stakeholders",
        "Deployment requires Python version coordination across two environments",
    ],
}

elt_architecture = {
    "name": "Proposed ELT (dbt-based)",
    "steps": [
        "1. Fivetran (SaaS) loads raw watch_events to BigQuery raw layer (no transformation)",
        "2. dbt models define transformation rules as SQL",
        "3. dbt runs on schedule, creates marts.watch_events from raw layer",
    ],
    "pros": [
        "Raw data always preserved — any transformation bug is fixable by re-running dbt",
        "SQL transformations readable by analysts and reviewable in PR",
        "Business rule changes = updating one dbt model + git commit",
        "BigQuery is fast at SQL transformations — often faster than Python ETL",
    ],
    "cons": [
        "More raw data stored (cost consideration at 5M events/day)",
        "PII (if any) must be handled before reaching the warehouse",
        "SQL is less expressive than Python for very complex feature engineering",
    ],
}

for arch in [etl_architecture, elt_architecture]:
    print(f"\n=== {arch['name']} ===")
    for step in arch["steps"]:
        print(f"  {step}")
    print("Pros:")
    for p in arch["pros"]:
        print(f"  + {p}")
    print("Cons:")
    for c in arch["cons"]:
        print(f"  - {c}")

In [ ]:
# CinemaStream decision: Hybrid approach
hybrid_decision = {
    "ingest": "Load raw events to BigQuery raw layer (ELT — Fivetran handles ingestion)",
    "core_transform": "dbt handles business-logic transformations in SQL (ELT)",
    "ml_features": "Python pipeline extracts from BigQuery and creates ML features (ETL pattern for complex features)",
    "pii_handling": "Mask user emails in the extraction layer before they reach the warehouse (ETL for PII)",
    "rationale": "The 80% of analytics use cases are well-served by ELT + dbt. ML feature engineering needs Python. PII must be masked before storage.",
}

print("CinemaStream Architecture Decision:")
for k, v in hybrid_decision.items():
    print(f"  {k}: {v}")

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")

# Step E+L: load raw
raw = pd.read_csv("cinemastream/data/watch_events.csv", encoding="utf-8")
raw.to_sql("raw_events", conn, if_exists="replace", index=False)
print(f"[E+L] Raw events loaded: {len(raw)} rows")

# Step T: transform in SQL
conn.execute("""
CREATE TABLE staging_events AS
SELECT
    event_id,
    user_id,
    movie_id,
    watch_started,
    watch_minutes,
    completed,
    TRIM(device)  AS device,
    country,
    CASE
        WHEN watch_minutes < 30 THEN 'short'
        WHEN watch_minutes BETWEEN 30 AND 89 THEN 'medium'
        ELSE 'long'
    END AS session_bucket
FROM raw_events
WHERE watch_minutes >= 1
""")
conn.commit()

staged_count = pd.read_sql("SELECT COUNT(*) AS n FROM staging_events", conn)['n'][0]
print(f"[T]   Staged events: {staged_count} rows")
sample = pd.read_sql("SELECT event_id, watch_minutes, session_bucket FROM staging_events LIMIT 3", conn)
print(sample)

---

# Chapter 45: Batch vs Streaming — When Data Moves

## 0. Where You Are

## 1. The Concept

## 2. Theory & Mechanics

In [ ]:
!pip install pandas

### Simulating Batch Processing

In [ ]:
import pandas as pd
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

# Batch job: process all events from a given date partition
def run_daily_batch(events_df: pd.DataFrame, batch_date: str) -> dict:
    """
    Process one day's worth of watch events.
    Returns a summary dict for logging.
    """
    start = datetime.utcnow()
    
    # Filter to this batch's date partition
    batch_events = events_df[
        events_df["watch_started"].str.startswith(batch_date)
    ].copy()
    
    if len(batch_events) == 0:
        return {"batch_date": batch_date, "rows": 0, "status": "empty_batch"}
    
    # Compute daily aggregates
    daily_stats = {
        "batch_date":       batch_date,
        "total_events":     len(batch_events),
        "unique_users":     batch_events["user_id"].nunique(),
        "avg_watch_min":    round(batch_events["watch_minutes"].mean(), 1),
        "completion_rate":  round(batch_events["completed"].mean() * 100, 1),
        "status":           "success",
        "processed_at":     start.isoformat(),
    }
    return daily_stats

events_df = pd.read_csv(
    "cinemastream/data/watch_events.csv",
    encoding="utf-8",
    parse_dates=["watch_started"],
)
events_df["watch_started"] = events_df["watch_started"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Simulate running daily batches
for date_str in ["2023-01-01", "2023-01-15", "2023-02-01"]:
    result = run_daily_batch(events_df, date_str[:10])
    print(f"Batch {date_str}: {result}")

### Simulating Stream Processing

In [ ]:
import time
import queue
import threading
from typing import Generator

def simulate_event_stream(events_df: pd.DataFrame, events_per_second: int = 5) -> Generator:
    """
    Simulate a real-time event stream by yielding events with small delays.
    In production: Kafka consumer, Kinesis stream, etc.
    """
    for _, event in events_df.head(15).iterrows():
        yield event.to_dict()
        time.sleep(1.0 / events_per_second)  # rate-limit

# In-memory "materialised view" that the stream processor maintains
running_stats = {
    "total_events":   0,
    "total_minutes":  0,
    "unique_users":   set(),
    "completed":      0,
}

def process_event_stream(events_df: pd.DataFrame) -> None:
    """Process events one at a time as they arrive."""
    print("Stream processor started...")
    
    for i, event in enumerate(simulate_event_stream(events_df, events_per_second=50)):
        # Update running stats
        running_stats["total_events"]  += 1
        running_stats["total_minutes"] += event.get("watch_minutes", 0) or 0
        running_stats["unique_users"].add(event["user_id"])
        if event.get("completed"):
            running_stats["completed"] += 1
        
        # "Emit" a real-time metric every 5 events
        if (i + 1) % 5 == 0:
            avg_min = running_stats["total_minutes"] / running_stats["total_events"]
            rate    = running_stats["completed"] / running_stats["total_events"] * 100
            print(f"  [stream @ event {i+1}] events={running_stats['total_events']} "
                  f"| users={len(running_stats['unique_users'])} "
                  f"| avg_min={avg_min:.1f} "
                  f"| completion={rate:.1f}%")

process_event_stream(events_df)

### Key Concepts: Windows and Watermarks

In [ ]:
# Simulating windowed aggregation in Python

from collections import defaultdict
from datetime import datetime, timedelta

def tumbling_window_agg(events: list[dict], window_minutes: int = 60) -> list[dict]:
    """
    Tumbling window: non-overlapping fixed-size time buckets.
    Each event belongs to exactly one window.
    In production: Flink / Spark Streaming / Kafka Streams handle this natively.
    """
    buckets = defaultdict(lambda: {"events": 0, "minutes": 0, "completed": 0})
    
    for event in events:
        if not event.get("watch_started"):
            continue
        dt = datetime.fromisoformat(event["watch_started"])
        # Round down to window boundary
        minutes_since_epoch = int(dt.timestamp() / 60)
        window_start_epoch  = (minutes_since_epoch // window_minutes) * window_minutes
        window_key = datetime.fromtimestamp(window_start_epoch * 60).strftime("%Y-%m-%d %H:00")
        
        bucket = buckets[window_key]
        bucket["events"] += 1
        bucket["minutes"] += event.get("watch_minutes", 0) or 0
        bucket["completed"] += 1 if event.get("completed") else 0
    
    results = [
        {
            "window": window_key,
            "events": b["events"],
            "avg_min": round(b["minutes"] / b["events"], 1) if b["events"] else 0,
            "completion_pct": round(b["completed"] / b["events"] * 100, 1) if b["events"] else 0,
        }
        for window_key, b in sorted(buckets.items())
    ]
    return results

# Run windowed aggregation on first 30 events
sample_events = events_df.head(30).to_dict("records")
windows = tumbling_window_agg(sample_events, window_minutes=60)
print("Hourly tumbling window stats:")
for w in windows[:6]:
    print(f"  {w}")

## 3. CinemaStream in Practice

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime

DATA_DIR = Path("cinemastream/data")
users_df  = pd.read_csv(DATA_DIR / "users.csv",        encoding="utf-8")
events_df = pd.read_csv(DATA_DIR / "watch_events.csv", encoding="utf-8")

# CinemaStream Data Flow 1: Weekly business metrics (BATCH — appropriate)
# - Consumers: Rohan, Dharani, board deck
# - SLA: available by 09:00 Monday morning
# - Latency acceptable: 7 days (weekly batch)
# - Complexity: low (SQL + Airflow scheduler)

def weekly_business_batch(users: pd.DataFrame, events: pd.DataFrame) -> dict:
    """Weekly batch job: compute key business metrics."""
    active   = users[users["churned"] == False]
    total_mrr = (active["plan"].map({"Free": 0, "Basic": 8.90, "Premium": 12.90})).sum()
    
    return {
        "report_type":    "WEEKLY_BATCH",
        "run_time":       datetime.utcnow().isoformat(),
        "total_users":    len(users),
        "active_users":   len(active),
        "mrr_sgd":        round(total_mrr, 2),
        "active_viewers": events["user_id"].nunique(),
        "avg_session_min": round(events["watch_minutes"].mean(), 1),
        "latency_target": "7 days (weekly Monday 09:00 SGT)",
        "appropriate":    True,
    }

weekly_report = weekly_business_batch(users_df, events_df)
for k, v in weekly_report.items():
    print(f"  {k}: {v}")

In [ ]:
# CinemaStream Data Flow 2: Real-time "Still Watching?" prompt (STREAMING — necessary)
# - Consumer: mobile app, to decide whether to show a "continue watching?" pop-up after 90 minutes
# - SLA: within 200ms of a user reaching 90 minutes of continuous viewing
# - Latency acceptable: < 1 second
# - Complexity: high (Kafka + Flink + Redis)
# This CANNOT be batch — a 1-hour batch run means users get the prompt 59 minutes late.

class StreamingUserSessionTracker:
    """
    Simplified in-memory tracker that simulates what a streaming processor
    (Flink, Kafka Streams) would maintain in a state store.
    """
    def __init__(self):
        self._sessions: dict[int, dict] = {}  # user_id → session state
    
    def process_event(self, event: dict) -> list[str]:
        """
        Process a single event. Returns list of actions to take.
        In production: this runs inside a Flink UDF or Kafka Streams topology.
        """
        user_id = event["user_id"]
        actions = []
        
        if user_id not in self._sessions:
            self._sessions[user_id] = {"total_minutes": 0, "events": 0, "prompted": False}
        
        session = self._sessions[user_id]
        session["total_minutes"] += event.get("watch_minutes", 0)
        session["events"] += 1
        
        # Trigger "still watching?" if user crosses 90-minute threshold and hasn't been prompted
        if session["total_minutes"] >= 90 and not session["prompted"]:
            session["prompted"] = True
            actions.append(f"SEND_PROMPT: user_id={user_id} has watched {session['total_minutes']}min")
        
        return actions

tracker = StreamingUserSessionTracker()

# Simulate Ravi Kumar's (user_id=1) events arriving in real-time
ravi_events = events_df[events_df["user_id"] == 1].head(5).to_dict("records")
print("Simulating real-time event stream for Ravi Kumar (user_id=1):")
for event in ravi_events:
    actions = tracker.process_event(event)
    state = tracker._sessions[1]
    print(f"  Event: {event['watch_minutes']}min → total={state['total_minutes']}min"
          + (f" → TRIGGER: {actions[0]}" if actions else ""))

In [ ]:
# Priya explains the architecture decision to the team:
architecture_decision = """
=== CinemaStream Batch vs Streaming Decision Matrix ===

Use Case                     | Latency      | Architecture | Tooling
-----------------------------|--------------|--------------|-------
Weekly business metrics             | 7 days       | Batch        | Airflow + SQL
Daily churn risk report             | 24 hours     | Batch        | Airflow + dbt
ML model training                   | Weekly       | Batch        | Airflow + Python
User recommendation refresh         | 1 hour       | Batch        | Airflow + Python
"Still watching?" prompt            | < 1 second   | Streaming    | Kafka + Flink
Real-time content performance       | < 5 minutes  | Micro-batch  | Spark Structured Streaming
Fraud detection (billing anomaly)   | < 10 seconds | Streaming    | Kafka Streams
Email for inactive users (weekly)   | 24 hours     | Batch        | Airflow + SendGrid

Key rule: Only use streaming if sub-minute latency is a HARD business requirement.
          Streaming is 5-10x harder to operate than batch.
"""
print(architecture_decision)

## 4. Pitfalls & Pro Tips

## 5. Exercises

In [ ]:
from datetime import datetime, timedelta
import pandas as pd

def sliding_window_events(events_df: pd.DataFrame,
                          window_hours: int = 24,
                          step_hours: int = 6) -> list[dict]:
    df = events_df.copy()
    df["watch_started"] = pd.to_datetime(df["watch_started"])
    df = df.dropna(subset=["watch_started"]).sort_values("watch_started")
    
    if df.empty:
        return []
    
    start = df["watch_started"].min().replace(hour=0, minute=0, second=0)
    end   = df["watch_started"].max()
    results = []
    
    window_start = start
    while window_start < end:
        window_end = window_start + timedelta(hours=window_hours)
        window_data = df[(df["watch_started"] >= window_start) &
                         (df["watch_started"] < window_end)]
        results.append({
            "window_start":  window_start.strftime("%Y-%m-%d %H:00"),
            "window_end":    window_end.strftime("%Y-%m-%d %H:00"),
            "event_count":   len(window_data),
            "avg_min":       round(window_data["watch_minutes"].mean(), 1) if len(window_data) > 0 else 0,
        })
        window_start += timedelta(hours=step_hours)
    
    return results

events_df2 = pd.read_csv("cinemastream/data/watch_events.csv", encoding="utf-8")
windows = sliding_window_events(events_df2, window_hours=24*7, step_hours=24)
for w in windows[:4]:
    print(w)

In [ ]:
from collections import defaultdict
from datetime import datetime, timedelta

class BingeWatchDetector:
    """
    Streaming processor: detects binge watching patterns.
    Maintains per-user session state in memory.
    In production: state stored in Redis or Flink state backend.
    """
    BINGE_THRESHOLD    = 3      # completed events
    WINDOW_HOURS       = 2      # hours
    COOLDOWN_HOURS     = 4      # minimum time before sending another notification

    def __init__(self):
        self._user_state: dict[int, dict] = defaultdict(lambda: {
            "completed_events": [],
            "last_notified": None,
        })

    def process_event(self, event: dict) -> list[str]:
        if not event.get("completed"):
            return []

        user_id = event["user_id"]
        state   = self._user_state[user_id]
        now     = datetime.fromisoformat(event.get("watch_started", "2024-01-01"))

        # Keep only events within the 2-hour window
        cutoff = now - timedelta(hours=self.WINDOW_HOURS)
        state["completed_events"] = [
            t for t in state["completed_events"] if t > cutoff
        ]
        state["completed_events"].append(now)

        # Check binge threshold
        actions = []
        if len(state["completed_events"]) >= self.BINGE_THRESHOLD:
            # Check cooldown
            last = state["last_notified"]
            if last is None or (now - last) > timedelta(hours=self.COOLDOWN_HOURS):
                state["last_notified"] = now
                actions.append(
                    f"NOTIFICATION: user_id={user_id} watched "
                    f"{len(state['completed_events'])} titles in 2hrs — 'Take a break!'"
                )
        return actions


# Test with a simulated event sequence for user 1
detector = BingeWatchDetector()
test_events = [
    {"user_id": 1, "completed": True,  "watch_started": "2024-01-01 19:00:00", "watch_minutes": 90},
    {"user_id": 1, "completed": True,  "watch_started": "2024-01-01 20:35:00", "watch_minutes": 88},
    {"user_id": 2, "completed": True,  "watch_started": "2024-01-01 20:40:00", "watch_minutes": 95},
    {"user_id": 1, "completed": False, "watch_started": "2024-01-01 21:00:00", "watch_minutes": 45},
    {"user_id": 1, "completed": True,  "watch_started": "2024-01-01 21:10:00", "watch_minutes": 86},  # binge!
    {"user_id": 2, "completed": True,  "watch_started": "2024-01-01 22:00:00", "watch_minutes": 92},
    {"user_id": 2, "completed": True,  "watch_started": "2024-01-01 22:50:00", "watch_minutes": 88},  # binge!
]

for event in test_events:
    actions = detector.process_event(event)
    if actions:
        print(f"Triggered at {event['watch_started']}: {actions[0]}")
    else:
        uid = event['user_id']
        n = len(detector._user_state[uid]['completed_events']) if event.get('completed') else '-'
        print(f"  {event['watch_started']} user_id={uid} completed={event['completed']} window_count={n}")